## Pipeline Modification — Sign Correction in System Jacobian

**Date:** 2026-09-09
**Status:** Fix identified, validation in progress

### Issue

While cross-checking the formulation against Plante (2022) and related
literature, a sign error was identified in the definition of the system
Jacobian used for the global stability eigenvalue problem.

**Correct definition:**

$$A = -V^{-1} \frac{dR}{dU}\Big|_{U_0}$$

evaluated at the base flow $U_0$, where $V$ is the diagonal matrix of
control-volume weights.

**What the pipeline actually produced:** `flux_jacobian_assembly_v4` and
`v5` assemble $V^{-1} \, dR/dU$ — the negative sign was missing.

### Fix

Negate the Jacobian immediately after loading, before conversion to
PETSc format:

```python
J_csr = read_jacobian(JACOBIAN_PATH)
J_csr = -J_csr          # apply missing negative sign: A = -V^-1 dR/dU
A = scipy_csr_to_petsc(J_csr)
```

### Expected effect on results

Since $A' = -A$, every eigenvalue's sign flips: $\lambda(A') = -\lambda(A)$.

- Growth rate **and** frequency both change sign.
- Eigenvectors are unchanged.
- The Re at which a mode crosses $\mathrm{Re}(\lambda)=0$ does not move —
  only which side is labeled stable vs. unstable flips.

### Validation plan

- [ ] Re-run cylinder case (`eigensolver_v1`) with the corrected sign.
- [ ] Re-run OAT15 case with corrected sign.
- [ ] Flag/relabel any prior result `.npz` files generated before this
      fix, since their stable/unstable labeling may be inverted.

### Affected files

- `flux_jacobian_assembly_v4` — assembler, pre-fix
- `flux_jacobian_assembly_v5` — assembler, pre-fix
- All eigendata `.npz` outputs generated prior to this fix

In [5]:
"""
Global flux Jacobian stability analysis pipeline.

1. Reads a sparse Jacobian saved in scipy's .npz (CSR) format.
2. Solves the eigenproblem with SLEPc shift-invert, sigma settable
   as a complex number, targeting eigenvalues near sigma (robust
   convergence for interior/near-marginal modes).
3. Reports the 10 eigenvalues with the largest growth rate among
   those found, with diagnostics to help you judge whether to
   trust each one.

Requires the complex-scalar PETSc/SLEPc build (petsc-complex env),
since sigma is a general complex number here.
"""
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from petsc4py import PETSc
from slepc4py import SLEPc



# =====================================================================
# 1. Read the Jacobian
# =====================================================================
def read_jacobian(path):
    """Load a sparse Jacobian saved via scipy.sparse.save_npz.
    Returns a CSR matrix cast to PETSc's scalar type (complex128
    in this environment)."""
    J = sp.load_npz(path)
    J = J.tocsr().astype(PETSc.ScalarType)
    print(f"Loaded Jacobian: shape={J.shape}, nnz={J.nnz}, "
          f"density={J.nnz / (J.shape[0]*J.shape[1]):.2e}")
    return J


def scipy_csr_to_petsc(J_csr):
    Mat = PETSc.Mat().createAIJ(size=J_csr.shape,
                                  csr=(J_csr.indptr, J_csr.indices, J_csr.data))
    Mat.assemble()
    return Mat


# =====================================================================
# 2. Solve with shift-invert
# =====================================================================
def solve_shift_invert(J, sigma, nev=10, ncv=None, tol=1e-10, max_it=2000):
    """
    sigma  : complex shift -- place this near where you expect the
             largest-growth-rate / marginal eigenvalues to sit
             (e.g. 0+0j, or 0 + 1j*omega_guess if you have a frequency
             estimate from a prior run or physical intuition).
    nev    : number of eigenvalues requested.
    ncv    : Krylov subspace size. Larger = more robust convergence,
             especially for clustered eigenvalues near a Hopf crossing,
             at the cost of more memory/compute per iteration. A common
             rule of thumb is ncv >= 2*nev, with more headroom (3-4x)
             if you expect closely spaced eigenvalues. Default here
             picks max(3*nev, 30), capped at the matrix dimension.
    """
    n = J.getSize()[0]
    if ncv is None:
        ncv = min(n, max(3 * nev, 30))

    E = SLEPc.EPS().create()
    E.setOperators(J)
    E.setProblemType(SLEPc.EPS.ProblemType.NHEP)  # non-symmetric flux Jacobian
    E.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    E.setDimensions(nev=nev, ncv=ncv)
    E.setTolerances(tol=tol, max_it=max_it)

    st = E.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(sigma)

    ksp = st.getKSP()
    ksp.setType('preonly')
    pc = ksp.getPC()
    pc.setType('lu')
    try:
        pc.setFactorSolverType('mumps')
        solver_used = 'mumps'
    except PETSc.Error:
        pc.setFactorSolverType('petsc')
        solver_used = 'petsc (built-in, no mumps found)'

    E.setTarget(sigma)
    E.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE)
    E.setFromOptions()

    print(f"Solving: sigma={sigma}, nev={nev}, ncv={ncv}, "
          f"factorization={solver_used}")
    E.solve()
    return E


# =====================================================================
# 3. Extract and report results
# =====================================================================
def report_results(E, J, nev, residual_tol=1e-6):
    nconv = E.getConverged()
    nev_requested = E.getDimensions()[0]

    print(f"\nConverged eigenpairs: {nconv} / {nev_requested} requested")
    if nconv < nev_requested:
        print("  WARNING: fewer eigenpairs converged than requested.")
        print("  Consider: increasing ncv, increasing max_it, or checking")
        print("  whether sigma is placed sensibly relative to the spectrum.")

    vr, vi = J.createVecs()
    results = []
    for i in range(nev):
        val = E.getEigenpair(i, vr, vi)
        err = E.computeError(i)
        results.append({
            'eigenvalue': val,
            'residual': err,
            'vec_real': vr.getArray().copy(),
            'vec_imag': vi.getArray().copy(),
        })

    # sort by largest growth rate (real part), descending
    # results.sort(key=lambda r: -r['eigenvalue'].real)
    top_results = results[:nev]

    print(f"\n{'#':>3} {'Re(lambda)':>14} {'Im(lambda)':>14} {'residual':>12}  status")
    print("-" * 66)
    for i, r in enumerate(top_results):
        lam = r['eigenvalue']
        err = r['residual']
        if err > residual_tol:
            status = "UNRELIABLE (residual above tol)"
        elif lam.real > 0:
            status = "UNSTABLE"
        elif abs(lam.real) < 1e-3:
            status = "MARGINAL (near Re=0 -- check carefully)"
        else:
            status = "stable"
        print(f"{i:>3} {lam.real:>14.6f} {lam.imag:>14.6f} {err:>12.2e}  {status}")

    return top_results


# =====================================================================
# Print ALL converged eigenvalues (not just top 10)
# =====================================================================
def print_eigenvalues(results, residual_tol=1e-6):
    print(f"\n{len(results)} eigenvalues:")
    print(f"{'#':>3} {'Re(lambda)':>14} {'Im(lambda)':>14} {'residual':>12}  status")
    print("-" * 66)
    for i, r in enumerate(results):
        lam = r['eigenvalue']
        err = r['residual']
        if err > residual_tol:
            status = "UNRELIABLE (residual above tol)"
        elif lam.real > 0:
            status = "UNSTABLE"
        elif abs(lam.real) < 1e-3:
            status = "MARGINAL (near Re=0)"
        else:
            status = "stable"
        print(f"{i:>3} {lam.real:>14.6f} {lam.imag:>14.6f} {err:>12.2e}  {status}")


# =====================================================================
# Plot ALL converged eigenvalues on the complex plane
# =====================================================================
def plot_eigenspectrum(results, residual_tol=1e-6, save_path='eigenspectrum.png'):
    eigs = np.array([r['eigenvalue'] for r in results])
    residuals = np.array([r['residual'] for r in results])
    reliable = residuals <= residual_tol

    fig, ax = plt.subplots(figsize=(7.5, 6))

    ax.axvspan(min(eigs.real.min(), -0.1) - 0.05, 0, color='tab:blue', alpha=0.06)
    ax.axvspan(0, max(eigs.real.max(), 0.1) + 0.05, color='tab:red', alpha=0.06)
    ax.axvline(0, color='black', lw=1.0)

    ax.scatter(eigs.real[reliable], eigs.imag[reliable],
               c='tab:blue', s=45, edgecolor='white', linewidth=0.5,
               label='converged (residual OK)', zorder=3)
    if (~reliable).any():
        ax.scatter(eigs.real[~reliable], eigs.imag[~reliable],
                   c='gray', s=45, marker='x',
                   label='residual above tolerance -- do not trust', zorder=3)

    # annotate the leading (largest growth rate) eigenvalue
    idx_lead = np.argmax(eigs.real)
    ax.annotate(f'lambda = {eigs[idx_lead].real:.4f} + {eigs[idx_lead].imag:.4f}j',
                xy=(eigs[idx_lead].real, eigs[idx_lead].imag),
                xytext=(10, 10), textcoords='offset points', fontsize=8)

    ax.set_xlabel('Re(lambda)  (growth rate)')
    ax.set_ylabel('Im(lambda)  (frequency, rad/s or non-dim)')
    ax.set_title('Eigenspectrum of the global flux Jacobian')
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"\nSaved: {save_path}")

# =====================================================================
# Save eigenvalues + eigenvectors in a portable format (.npz)
# =====================================================================
def save_eigendata(E, J, sigma, out_path='eigendata.npz'):
    """
    Saves:
      eigenvalues : complex array, shape (nconv,)
      eigenvectors: complex array, shape (nconv, N) -- each row is one
                    eigenvector, full length N (matches Jacobian dimension)
      residuals   : real array, shape (nconv,)
      sigma       : the complex shift used for this solve
      nev, ncv    : solver settings used, for reproducibility
    """
    nconv = E.getConverged()
    N = J.getSize()[0]
    nev_requested, ncv_used, _ = E.getDimensions()

    vr, vi = J.createVecs()
    eigenvalues = np.zeros(nconv, dtype=complex)
    eigenvectors = np.zeros((nconv, N), dtype=complex)
    residuals = np.zeros(nconv)

    for i in range(nconv):
        val = E.getEigenpair(i, vr, vi)
        eigenvalues[i] = val
        residuals[i] = E.computeError(i)
        # PETSc splits real/imag into separate vecs even in a complex
        # build when using getEigenpair this way -- combine them here
        eigenvectors[i, :] = vr.getArray() + 1j * vi.getArray()

    np.savez(out_path,
             eigenvalues=eigenvalues,
             eigenvectors=eigenvectors,
             residuals=residuals,
             sigma=np.array([sigma]),
             nev=nev_requested,
             ncv=ncv_used)

    print(f"Saved {nconv} eigenpairs to {out_path}")
    print(f"  eigenvalues.shape  = {eigenvalues.shape}")
    print(f"  eigenvectors.shape = {eigenvectors.shape}  (row i = eigenvector for eigenvalues[i])")
    
# =====================================================================
# Interpretation guide (printed, not just code) -- read this after running
# =====================================================================
def print_interpretation_guide():
    print("""
--------------------------------------------------------------------
How to interpret this output:

1. Re(lambda) > 0  -> that mode grows in time -> globally unstable.
   Re(lambda) < 0  -> decays -> stable.
   Re(lambda) ~ 0  -> marginal; this is the Hopf-relevant regime.

2. A genuine Hopf bifurcation shows up as a COMPLEX-CONJUGATE PAIR
   (nonzero Im(lambda), and you should see its conjugate elsewhere
   in the list or in a re-run with wider nev) with Re(lambda) crossing
   zero as you vary your control parameter (Reynolds number). A real
   eigenvalue crossing zero alone would indicate a different
   (steady/pitchfork) bifurcation, not Hopf.

3. Trust ONLY eigenpairs with residual comfortably below your solver
   tolerance (1e-6 to 1e-8 is typical). An eigenvalue with residual
   above tolerance is not converged -- don't draw physical conclusions
   from it, especially near Re(lambda)=0 where you need real precision.

4. If nconv < nev requested: SLEPc could not converge everything you
   asked for within max_it iterations at this ncv. Don't assume the
   unconverged ones don't exist -- widen ncv/max_it and re-run before
   concluding anything about the missing eigenvalues.

5. Sanity checks before trusting a Hopf conclusion:
   - Increase ncv and re-run: do the top eigenvalues change more than
     your tolerance? If yes, ncv was too small.
   - Nudge sigma slightly and re-run: do the same eigenvalues reappear?
     If eigenvalues disappear/appear with small sigma changes, you may
     be missing modes near the edge of what shift-invert "saw".
   - If you have a mesh-refinement study available, confirm the
     leading eigenvalue's real part doesn't move significantly under
     refinement -- a Hopf point that moves with mesh resolution isn't
     trustworthy yet.
--------------------------------------------------------------------
""")

In [6]:
# read the jacobian
Mesh = 21228
Re   = 60

Mach = 0.2
# AoA = 35

gamma = 1.4
R_gas = 287.0          # confirm units match the solver

# define path
# data_dir = "/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v4/v3_mesh"

data_dir = "../../data/flux_jacobian_assembly_v4/v1_mesh"
# data_dir = "./data/flux_jacobian"
JACOBIAN_PATH = f"{data_dir}/jacobian_cylinder_{Mesh}_Re{Re}_M{Mach}_fd.npz"   # <-- set to your actual file

# JACOBIAN_PATH = f"{data_dir}/jacobian_OAT15_M0.73_A35_fd_harten0.05.npz"

J_csr = read_jacobian(JACOBIAN_PATH)

A = -J_csr #adding negative sign at the front to define the system jacobian matrix A

# convert CSR matrix to petsc compatible form + adding negative sign at the front to define the system jacobian matrix A
A = scipy_csr_to_petsc(A)

Loaded Jacobian: shape=(106140, 106140), nnz=5961200, density=5.29e-04


In [ ]:
import time

# set the shift SIGMA
f = 9.505 # frequency in Hz
# f = 17.67 # frequency in Hz from St = 0.07 for OAT15

SIGMA = 0.0 + f * 2 * np.pi*1j                            # <-- set your shift here

nev = 10
ncv = 300
t0 = time.perf_counter()

E = solve_shift_invert(A, sigma=SIGMA, nev=nev, ncv = ncv)
# save run time 
t = time.perf_counter() - t0
top_results = report_results(E,A, nev) # reporting the top 10 eigenvalues that are close to SIGMA
print(f"Run time: {t:.2f}s")

Solving: sigma=59.72167634474197j, nev=10, ncv=300, factorization=mumps

Converged eigenpairs: 105 / 10 requested



  #     Re(lambda)     Im(lambda)     residual  status
------------------------------------------------------------------
  0     -12.798011      51.499954     3.78e-06  UNRELIABLE (residual above tol)
  1     -15.593963      59.600956     2.41e-06  UNRELIABLE (residual above tol)
  2      -7.120113      45.683439     8.68e-07  stable
  3     -15.074399      55.132057     1.85e-06  UNRELIABLE (residual above tol)
  4     -18.209987      68.287392     7.05e-06  UNRELIABLE (residual above tol)
  5     -12.618026      43.133807     2.74e-06  UNRELIABLE (residual above tol)
  6     -22.859038      66.797208     4.49e-06  UNRELIABLE (residual above tol)
  7     -24.129423      52.674993     7.68e-06  UNRELIABLE (residual above tol)
  8     -18.106596      41.517724     1.01e-05  UNRELIABLE (residual above tol)
  9     -10.671825      34.723170     7.72e-06  UNRELIABLE (residual above tol)
Run time: 37.45s


In [9]:
# save eigenvectors and eigenvalues
# out_dir = "./data/ncv_sweep"

# out_dir = "/home/ahf25/git/flux_jacobian/data/eigendata/OAT15"
out_dir = "/home/ahf25/git/flux_jacobian/data/eigendata/eigensolver_v3/v1_mesh"

eigen_file = f"eigendata_{Mesh}_Re{Re}_M{Mach}_nev{nev}_ncv{ncv}.npz"

# eigen_file = f"eigendata_M{Mach}_A{AoA}_nev{nev}_ncv{ncv}.npz"

save_eigendata(E, A, sigma=SIGMA, out_path=f"{out_dir}/{eigen_file}")


Saved 105 eigenpairs to /home/ahf25/git/flux_jacobian/data/eigendata/eigensolver_v3/v1_mesh/eigendata_21228_Re60_M0.2_nev10_ncv300.npz
  eigenvalues.shape  = (105,)
  eigenvectors.shape = (105, 106140)  (row i = eigenvector for eigenvalues[i])
